# Run the full benchmark on Colab (GPU)

Steps: GPU on → get code → install → data → token → run → report → download.

**Before the first run:** in Colab, open the 🔑 *Secrets* panel (left sidebar) and add
a secret named `HF_TOKEN` with your Hugging Face read token, and switch on
"Notebook access". (Hugging Face = the site hosting TabPFN's model weights.
You must also accept the terms at https://huggingface.co/Prior-Labs/tabpfn_3_5 once.)

## 1. Check the GPU is on
Runtime → Change runtime type → T4 GPU. This cell should print a GPU name.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Get the code from GitHub
Replace the URL with your repo's URL.

In [ ]:
REPO_URL = "https://github.com/YOUR-USERNAME/small-tabular-benchmark.git"  # <-- edit
!git clone -q {REPO_URL} project
%cd project

## 3. Install packages
Uses the same pinned versions as the laptop.

In [ ]:
!pip install -q -r requirements.txt

## 4. Add the data files
Upload the four files from your laptop's `data/` folder:
`winequality-red.csv`, `winequality-white.csv`, `adult.data`, `adult.test`.

In [ ]:
from google.colab import files
import shutil
uploaded = files.upload()          # opens a file picker
for name in uploaded:
    shutil.move(name, f"data/{name}")
!ls data

## 5. Give TabPFN your Hugging Face token (read from Colab Secrets, never typed into the notebook)

In [ ]:
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

## 6. Save results to Google Drive (survives disconnects)

Results are written to Drive after every model fit. If Colab disconnects, re-run
cells 1–6 and then cell 7: finished work is skipped automatically.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/small-tabular-benchmark/raw_results.csv"
os.makedirs(os.path.dirname(OUT), exist_ok=True)

## 7. Quick check first (a few minutes), then the full run

In [ ]:
!python -m src.experiment --quick --out /content/drive/MyDrive/small-tabular-benchmark/raw_results_quick.csv

In [ ]:
!python -m src.experiment --out {OUT}

## 8. Make the summary tables and plots

In [ ]:
!python -m src.report --raw {OUT}
!ls results/tables results/figures

## 9. Download results
Unzip into your laptop's project folder, then commit them with git.

In [ ]:
!cp {OUT} results/tables/raw_results.csv
!zip -qr results.zip results
files.download("results.zip")